<a href="https://colab.research.google.com/github/chetankumarmk56/Claude-Agentic-SDK-Labs/blob/main/Lab-2/Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Lab 2 — DB Operations
### PostgreSQL on Neon + FastAPI + Streamlit, with an AI mode

In this lab you will build and run an **insert-only** database application over
three tables — `customer`, `product` and `sales` — and then use it two ways:

| | Mode | What it does |
|---|---|---|
| **Manual** | Forms and tables | You type the values. Every write goes through a validated API. |
| **AI** | Chat | You ask in plain language. Claude calls the same API endpoints as tools — it never writes SQL. |

The rule that shapes everything: **SELECT and INSERT only. No UPDATE, no
DELETE, anywhere.** A sale is a historical fact — once recorded, its price and
its time never change, even when the product's price changes later.

That constraint is what makes the lab checkable. An append-only table can be
rebuilt from scratch and compared against a previous run without ambiguity,
which is exactly what the last two steps do.


## Step 1 — Clone the repository and install dependencies

Run the cell below. It clones only this lab and installs the Python packages.

If you see a **Restart session** dialog while it installs, click **Restart
session**, then run this cell again.

pip will print a red `dependency resolver` warning about `google-adk`,
`tenacity` and `watchdog`. That is Colab's own pre-installed packages
disagreeing with Streamlit's pins — nothing this lab uses. Ignore it; the
three lines the cell prints at the end are what tell you the install
worked.


In [ ]:
# Start clean. Without this, a second run of this cell hits
# "destination path already exists", git clone fails, and every cell below
# silently keeps running against the OLD copy of the lab.
%cd /content
!rm -rf /content/Claude-Agentic-SDK-Labs

# Clone only what this lab needs
# --depth 1: only the latest commit          --filter=blob:none: skip file contents
# --sparse: check out one folder rather than the whole repository
!git clone --depth 1 --filter=blob:none --sparse https://github.com/chetankumarmk56/Claude-Agentic-SDK-Labs.git
%cd /content/Claude-Agentic-SDK-Labs
!git sparse-checkout set Lab-2
%cd /content/Claude-Agentic-SDK-Labs/Lab-2

# Python dependencies for the lab
!pip install -q -r requirements.txt

# ngrok exposes the Streamlit app so you can open it from Colab
!pip install -q pyngrok

print("\nFetched commit:")
!git log --oneline -1
print("\nTables this lab creates:")
!grep -oE "^CREATE TABLE [a-z]+" db/schema.sql
print("\nReject reasons the loader can report:")
!grep -oE '^    [A-Z_]+ = "[A-Z_]+"' repository/errors.py | sed 's/.*= //;s/"//g'


## Step 2 — Enter your Neon connection string

The database lives on [Neon](https://console.neon.tech). Create a free project,
copy the connection string from **Connect**, and paste it below.

`getpass` hides what you type, so the connection string never appears in the
notebook output and is never saved with the file.

Paste it exactly as Neon gives it — `postgres://` and `postgresql://` both
work, and the driver is normalised for you.


In [ ]:
import os, getpass

os.environ["DATABASE_URL"] = getpass.getpass("Paste your Neon connection string: ")
print("Connection string stored for this session only.")


## Step 3 — Create the schema and seed it

`db/schema.sql` is the single source of truth for the database: three tables,
every CHECK, both foreign keys, the unique natural key on `sales`, the
generated `line_total` column and the four indexes. Nothing in the application
issues DDL — the SQLAlchemy models only mirror this file.

`db/drop.sql` runs first so the cell is safe to re-run.


In [ ]:
!python db/run.py drop.sql schema.sql seed.sql


## Step 4 — Prove the constraints are real

The database refuses bad data by itself, not merely the Python around it. This
is the check that matters: if validation lived only in the application, anything
reaching the database another way could still corrupt it.

The suite covers schema drift, `quantity` 0 and -1 refused **by the database**,
`line_total` always equal to `quantity * unit_price` and impossible to write
directly, a duplicate natural key raising rather than duplicating, and a
`Decimal` surviving a round trip through `NUMERIC`.


In [ ]:
os.environ["TEST_DATABASE_URL"] = os.environ["DATABASE_URL"]
!python -m pytest -q


## Step 5 — Load a real file through the loader

`reference/sample-sales.csv` holds 24 rows: **16 that must load and 8 built to
be rejected**, between them covering all six reject reasons.

Watch which rows come back rejected and why. Two of them are duplicates of rows
earlier in the same file — the natural key `(customer_id, product_id, sold_at)`
is what catches them, which is also what makes re-running the loader safe.


In [ ]:
!python scripts/load_sales.py reference/sample-sales.csv


### Re-run it, and nothing new goes in

The same file, loaded a second time. Every row that was accepted before is now
reported `DUPLICATE_SALE`; rows rejected for another reason keep that reason,
because they were never inserted and so cannot be duplicates.

**Zero rows are added.** That is idempotence, and it is a property of the
schema — not of the loader being careful.


In [ ]:
!python scripts/load_sales.py reference/sample-sales.csv


### Read it back out of the database

Not out of memory — out of `sales_detail`, the one view that joins all three
tables. It is ordered on natural keys (`sold_at`, then email, then SKU), never
on a surrogate id, because ids are not stable across a rebuild.


In [ ]:
import pandas as pd
from sqlalchemy import text
from repository.db import get_engine

with get_engine().connect() as conn:
    rows = conn.execute(text("SELECT * FROM sales_detail")).mappings().all()

df = pd.DataFrame(rows)
print(f"{len(df)} rows;  revenue = {df['line_total'].sum():,.2f}")
df.head(10)


## Step 6 — Run the web application

Both modes, behind a Streamlit interface. `app.py` starts the FastAPI backend as
a child process and then hands over to the UI, so one command runs the whole
thing behind a single tunnel.

The app asks for your **Neon connection string** and a **Claude API key** on its
own connect screen — the same run-time pattern as this notebook, in a UI.
Neither is written to disk.

Run the cell, then open the printed ngrok URL. **Leave this cell running** while
you use the app; stopping it closes the tunnel.


In [ ]:
import os, time, getpass, subprocess
from pyngrok import ngrok

PORT = 8501

# Stop anything an earlier run of this cell left behind.
#
# This matters more than it looks. Without it, a second run finds the port
# taken, the NEW server exits immediately, and the tunnel quietly keeps
# pointing at the OLD one — which shows up in the browser as "Please wait..."
# forever, with no obvious cause.
subprocess.run("pkill -f 'streamlit run'", shell=True)
subprocess.run("pkill -f 'uvicorn api.main'", shell=True)
ngrok.kill()
time.sleep(2)

# Streamlit asks for an email on first run, which stalls a headless start.
# Writing an empty credentials file skips the prompt.
os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)
with open(os.path.expanduser("~/.streamlit/credentials.toml"), "w") as f:
    f.write('[general]\nemail = ""\n')

# ngrok needs its own free authtoken - https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(getpass.getpass("Enter your ngrok authtoken: "))

# Open the tunnel first, then start the server behind it
public_url = ngrok.connect(PORT).public_url

get_ipython().system_raw(f"streamlit run app.py --server.port {PORT} > streamlit.log 2>&1 &")
time.sleep(15)

log = open("streamlit.log").read()
print("--- streamlit.log ---")
print(log[-1200:])

if "already in use" in log:
    print("\n!! The port was still busy, so the app did not start.")
    print("   Run this cell again - it clears the port first.")
else:
    print("\nOpen your app here:", public_url)
    print("Leave this cell running while you use the app; stopping it closes the tunnel.")
    time.sleep(60 * 20)


## Step 7 — Using the application

Open the ngrok URL and connect with your Neon string and a Claude API key.

**Manual mode** is the default. *Overview* shows live counts and revenue read
straight from the database. *Add* has the three insert forms. *Browse* has the
three list views plus the joined `sales_detail` view. *Import* takes a CSV and
reports accepted and rejected rows in separate tables, each rejection naming its
reason.

**AI mode** — the toggle at the top right — is the same database through a chat.
Try:

- `how many customers do we have?`
- `what has Ava bought?`
- `add a product MN-32C-OLED, 32-inch OLED, Displays, 899.00`
- `record a sale` — it will ask you which customer, which product, and when,
  rather than guessing

Claude reaches the database only through the API's own endpoints, given to it as
tools. It has no way to write SQL, so every guarantee above still holds no
matter what it decides to do. Ask it to delete something and it will tell you it
cannot — because nothing in the system can.


---

## What to take away

**Put the guarantee in the database, not the application.** A CHECK constraint
holds against every client, including the ones you have not written yet.
Validation in Python alone is a convention; validation in the schema is a rule.

**Append-only makes correctness checkable.** Because nothing is ever updated or
deleted, the same input always produces the same database — which is what lets
the lab rebuild from empty ten times and compare the results exactly.

**Money is `NUMERIC`, never a float.** Binary floating point cannot represent
0.10 exactly. A total that drifts by a cent is a defect, not a rounding style.

**A recorded fact keeps its own values.** The price on a sale is copied onto the
row, not read back through the product. Changing a price today must not rewrite
what happened last year.

**Give a model tools, not a database.** AI mode is useful precisely because it
cannot do anything the API would not let a form do. The safety lives in the
layer underneath, where it holds regardless of what the model decides.

**Never hardcode a credential.** The notebook prompts with `getpass`, the app
prompts with a password field. Neither is written to disk, and neither belongs
in a repository.
